# Pip 

In [ ]:

import pandas as pd
import numpy as np
from minio import Minio
from datetime import datetime
from multiprocessing import Pool, cpu_count
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from spacy.cli import download
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import io, json, re, nltk, spacy, faiss

# Bronze

In [ ]:

client = Minio("localhost:9000", "admin", "password123", secure=False)
bucket = "bigdata-nhom6"

if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

def push_bronze_kaggle(local_path, filename):
    path = f"bronze/kaggle/{filename}"
    client.fput_object(bucket, path, local_path)
    print(f"-> Bronze: {path}")

def push_bronze_crawl(data, filename):
    b = json.dumps(data).encode('utf-8')
    path = f"bronze/crawled/{datetime.now().strftime('%Y-%m-%d')}/{filename}"
    client.put_object(bucket, path, io.BytesIO(b), len(b), 'application/json')
    print(f"-> Bronze: {path}")

In [ ]:
# Đẩy file Kaggle
push_bronze_kaggle("/mnt/d/School/big_data/BigData_Nhom_6/storage/kaggledataset/job_skills.csv", "job_skills.csv")
push_bronze_kaggle("/mnt/d/School/big_data/BigData_Nhom_6/storage/kaggledataset/linkedin_job_postings.csv", "linkedin_job_postings.csv")

In [ ]:
# Đẩy data Crawl
with open("/mnt/d/School/big_data/BigData_Nhom_6/scraper/crawl_monster/monster_jobs_with_skills_chunk_1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

push_bronze_crawl(data, "monster_jobs_with_skills_chunk_1.json")

# 2. Silver

## 2.1 Khởi tạo Model NLP & Synonym Map

In [ ]:
try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    download("en_core_web_lg")
    nlp = spacy.load("en_core_web_lg")

for pkg in ['wordnet', 'omw-1.4', 'stopwords']:
    try:
        nltk.data.find(f'corpora/{pkg}')
    except:
        nltk.download(pkg, quiet=True)

lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))

SYNONYM_MAP = {
    "sr": "senior", "jr": "junior", "mid": "mid", "lead": "senior", "principal": "senior", "staff": "senior", "intern": "intern", "trainee": "junior",
    "mgr": "manager", "mngr": "manager", "dir": "director", "vp": "vice president", "svp": "senior vice president", "assoc": "associate", "asst": "assistant", "rep": "representative", "exec": "executive", "officer": "officer", "admin": "administrator",
    "dev": "developer", "eng": "engineer", "swe": "software engineer", "frontend": "front end", "backend": "back end", "fullstack": "full stack", "full-stack": "full stack", "qa": "quality assurance", "sdet": "software development engineer in test", "ml": "machine learning", "ai": "artificial intelligence", "ds": "data scientist", "de": "data engineer", "fe": "front end", "be": "back end",
    "analyst": "analyst", "analytics": "analyst", "bi": "business intelligence", "data analyst": "data analyst", "data analytics": "data analyst",
    "biz": "business", "bd": "business development", "bdr": "business development representative", "ba": "business analyst", "pm": "project manager", "pdm": "product manager",
    "ae": "account executive", "am": "account manager", "sales rep": "sales representative", "salesperson": "sales representative",
    "mkt": "marketing", "seo": "search engine optimization", "sem": "search engine marketing", "smm": "social media marketing",
    "acct": "accountant", "acc": "accountant", "cpa": "certified public accountant", "fp&a": "financial planning analysis",
    "hr": "human resources", "hrbp": "human resources business partner", "recruiter": "recruiter", "talent": "talent acquisition",
    "rn": "registered nurse", "lpn": "licensed practical nurse", "np": "nurse practitioner", "pa": "physician assistant", "md": "medical doctor", "dr": "doctor",
    "mech": "mechanical", "elec": "electrical", "civil": "civil", "arch": "architect", "tech": "technician",
    "ops": "operations", "coo": "chief operating officer", "logistics": "logistics", "supply chain": "supply chain",
    "prof": "professor", "lect": "lecturer", "ta": "teaching assistant",
    "cs": "customer service", "csr": "customer service representative", "support": "support",
    "&": "and", "/": " ", "pt": "part time", "ft": "full time"
}

## 2.2 Các hàm xử lý và làm sạch Text

In [ ]:
import io, json, re
import pandas as pd
import spacy
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Tải tài nguyên
try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    from spacy.cli import download
    download("en_core_web_lg")
    nlp = spacy.load("en_core_web_lg")

for pkg in ['wordnet', 'omw-1.4', 'stopwords']:
    try:
        nltk.data.find(f'corpora/{pkg}')
    except:
        nltk.download(pkg, quiet=True)

lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))

SYNONYM_MAP = {
    "sr": "senior", "jr": "junior", "mid": "mid", "lead": "senior", "principal": "senior", "staff": "senior", "intern": "intern", "trainee": "junior",
    "mgr": "manager", "mngr": "manager", "dir": "director", "vp": "vice president", "svp": "senior vice president", "assoc": "associate", "asst": "assistant", "rep": "representative", "exec": "executive", "officer": "officer", "admin": "administrator",
    "dev": "developer", "eng": "engineer", "swe": "software engineer", "frontend": "front end", "backend": "back end", "fullstack": "full stack", "full-stack": "full stack", "qa": "quality assurance", "sdet": "software development engineer in test", "ml": "machine learning", "ai": "artificial intelligence", "ds": "data scientist", "de": "data engineer", "fe": "front end", "be": "back end",
    "analyst": "analyst", "analytics": "analyst", "bi": "business intelligence", "data analyst": "data analyst", "data analytics": "data analyst",
    "biz": "business", "bd": "business development", "bdr": "business development representative", "ba": "business analyst", "pm": "project manager", "pdm": "product manager",
    "ae": "account executive", "am": "account manager", "sales rep": "sales representative", "salesperson": "sales representative",
    "mkt": "marketing", "seo": "search engine optimization", "sem": "search engine marketing", "smm": "social media marketing",
    "acct": "accountant", "acc": "accountant", "cpa": "certified public accountant", "fp&a": "financial planning analysis",
    "hr": "human resources", "hrbp": "human resources business partner", "recruiter": "recruiter", "talent": "talent acquisition",
    "rn": "registered nurse", "lpn": "licensed practical nurse", "np": "nurse practitioner", "pa": "physician assistant", "md": "medical doctor", "dr": "doctor",
    "mech": "mechanical", "elec": "electrical", "civil": "civil", "arch": "architect", "tech": "technician",
    "ops": "operations", "coo": "chief operating officer", "logistics": "logistics", "supply chain": "supply chain",
    "prof": "professor", "lect": "lecturer", "ta": "teaching assistant",
    "cs": "customer service", "csr": "customer service representative", "support": "support",
    "&": "and", "/": " ", "pt": "part time", "ft": "full time"
}

# BÍ KÍP TĂNG TỐC 1: Gộp toàn bộ từ đồng nghĩa thành 1 Pattern Regex duy nhất (Chạy nhanh hơn hàng trăm lần)
SYNONYM_PATTERN = re.compile(r'\b(' + '|'.join(re.escape(k) for k in SYNONYM_MAP.keys()) + r')\b')

def replace_synonyms(text):
    # Thay thế chỉ bằng 1 lần quét thay vì 50 lần lặp
    return SYNONYM_PATTERN.sub(lambda m: SYNONYM_MAP[m.group(0)], text)

In [ ]:
def fast_clean_text(text, max_words_per_phrase=5):
    if not text or not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"[/&\-]", " ", text)
    text = re.sub(r"[^\w\s,]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    phrases = [p.strip() for p in text.split(',') if p.strip()]
    cleaned_phrases = []
    
    for phrase in phrases:
        # Gọi hàm thay từ đồng nghĩa tốc độ cao
        phrase = replace_synonyms(phrase)
        
        # Lemmatize và xóa Stop words cùng lúc
        tokens = [lemmatizer.lemmatize(w) for w in phrase.split() if w not in STOP_WORDS]
        if tokens and len(tokens) <= max_words_per_phrase:
            cleaned_phrases.append(' '.join(tokens))
            
    # Xóa trùng lặp và nối chuỗi
    return ', '.join(dict.fromkeys(cleaned_phrases))

# BÍ KÍP TĂNG TỐC 2: spaCy Batch Processing
def process_text_batch(texts):
    results = []
    # Tắt parser, tagger, lemmatizer vì ta chỉ cần NER. n_process=-1 vắt kiệt tất cả CPU cores.
    for doc in nlp.pipe(texts, disable=["tagger", "parser", "attribute_ruler", "lemmatizer"], batch_size=2000, n_process=-1):
        try:
            text = doc.text
            if doc.ents:
                remove_spans = [(ent.start_char, ent.end_char) for ent in doc.ents if ent.label_ in ["ORG", "GPE", "LOC"]]
                for start, end in sorted(remove_spans, reverse=True):
                    text = text[:start] + text[end:]
                text = ' '.join(text.split())
            
            # Xử lý dọn rác ngay sau khi xóa NER
            results.append(fast_clean_text(text))
        except Exception:
            results.append("")
    return results

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, pandas_udf
from pyspark.sql.types import StringType
import pandas as pd

spark = SparkSession.builder \
    .appName("SilverLayer_NLP_Optimized") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

In [ ]:
@pandas_udf(StringType())
def spark_fast_clean_udf(texts: pd.Series) -> pd.Series:
    """Hàm này sẽ được Spark gọi và chạy song song trên nhiều lõi/máy"""
    results = []
    
    # nlp.pipe nhận trực tiếp lô dữ liệu từ Spark
    for doc in nlp.pipe(texts.astype(str), disable=["tagger", "parser", "attribute_ruler", "lemmatizer"], batch_size=2000):
        try:
            text = doc.text
            if doc.ents:
                remove_spans = [(ent.start_char, ent.end_char) for ent in doc.ents if ent.label_ in ["ORG", "GPE", "LOC"]]
                for start, end in sorted(remove_spans, reverse=True):
                    text = text[:start] + text[end:]
                text = ' '.join(text.split())
            
            results.append(fast_clean_text(text))
        except Exception:
            results.append("")
            
    return pd.Series(results)

In [ ]:
# Chuyển Pandas DataFrame sang Spark DataFrame
spark_df = spark.createDataFrame(df_silver)

print(f"Bắt đầu xử lý {spark_df.count()} dòng bằng Spark Cluster...")

# Áp dụng Pandas UDF
spark_df = spark_df.withColumn("job_title", spark_fast_clean_udf(col("job_title")))
spark_df = spark_df.withColumn("job_skills", spark_fast_clean_udf(col("job_skills")))

# Gộp chuỗi bằng hàm nội tại của Spark (cực nhanh)
spark_df = spark_df.withColumn("title_skills", concat_ws(" ", col("job_title"), col("job_skills")))

# Lọc bỏ dòng rỗng sau khi làm sạch
spark_df = spark_df.filter(col("title_skills").trim() != "")

# Tùy chọn 1: Chuyển lại thành Pandas để đẩy lên MinIO theo hàm cũ của bạn
df_clean_final = spark_df.toPandas()

# Tùy chọn 2 (Khuyến nghị cho Big Data): Ghi thẳng từ Spark xuống MinIO
# spark_df.write.csv("s3a://bigdata-nhom6/silver/Silver_Jobs_Cleaned.csv", header=True, mode="overwrite")

In [ ]:
def dedup_text(text):
    phrases = [p.strip() for p in text.split(',') if p.strip()]
    return ', '.join(dict.fromkeys(phrases))

def cut_noise(text, max_words_per_phrase=5):
    text = text.replace('.', ',').replace(';', ',')
    text = re.sub(r',+', ',', text) 
    phrases = [p.strip() for p in text.split(',') if p.strip()]
    phrases = [p for p in phrases if len(p.split()) <= max_words_per_phrase]
    return ', '.join(phrases)

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)  
    text = " ".join(text.split())
    return text

def remove_stop_words(text, stop_words=None):
    if stop_words is None or len(stop_words) == 0:
        stop_words = STOP_WORDS
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

def lemmatize(text):
    return " ".join(lemmatizer.lemmatize(w) for w in text.split())

def normalize_synonym_phrase(text):
    for k, v in SYNONYM_MAP.items():
        text = re.sub(rf"\b{k}\b", v, text)
    return text
    
def remove_org_loc(text):
    doc = nlp(text)
    remove_spans = [(ent.start_char, ent.end_char) for ent in doc.ents if ent.label_ in ["ORG", "GPE", "LOC"]]
    for start, end in sorted(remove_spans, reverse=True):
        text = text[:start] + text[end:]
    return ' '.join(text.split())
    
def clean_text(text, max_words_per_phrase=5):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"[/&\-]", " ", text)
    text = re.sub(r"[^\w\s,]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    phrases = [p.strip() for p in text.split(',') if p.strip()]
    cleaned_phrases = []
    
    for phrase in phrases:
        for k, v in SYNONYM_MAP.items():
            phrase = re.sub(rf"\b{k}\b", v, phrase)
        tokens = [w for w in phrase.split() if w not in STOP_WORDS]
        if not tokens: continue
        lemmatized = [lemmatizer.lemmatize(w) for w in tokens]
        if len(lemmatized) <= max_words_per_phrase:
            cleaned_phrases.append(' '.join(lemmatized))
            
    dedup_phrases = list(dict.fromkeys(cleaned_phrases))
    return ', '.join(dedup_phrases).strip(', ')

def process_text_wrapper(text):
    try:
        text = remove_org_loc(text) # Áp dụng xóa tên công ty/địa danh
        return clean_text(text)
    except: 
        return ""

## 2.3 Silver Layer - Merge & Clean Data

In [ ]:
import json
import pandas as pd
import io

df_meta = pd.read_csv(client.get_object(bucket, "bronze/kaggle/linkedin_job_postings.csv"))
df_skills = pd.read_csv(client.get_object(bucket, "bronze/kaggle/job_skills.csv"))

df_linkedin = pd.merge(df_meta, df_skills, on="job_link")[['job_link', 'job_title', 'job_skills']]

print(df_linkedin.count())
df_linkedin.head()

In [ ]:
import json
import pandas as pd

objs = list(client.list_objects(bucket, prefix="bronze/crawled/", recursive=True))
latest = sorted({o.object_name.split('/')[2] for o in objs if len(o.object_name.split('/')) >= 3}, reverse=True)[0]

monster_data_list = []

for o in objs:
    if o.object_name.startswith(f"bronze/crawled/{latest}/") and o.object_name.endswith(".json"):
        data = json.loads(client.get_object(bucket, o.object_name).read().decode())
        monster_data_list.extend(data if isinstance(data, list) else [data])


df_crawled = pd.DataFrame([
    {
        "job_link": row.get("link"),
        "job_title": row.get("title"),
        "job_skills": ", ".join(row.get("skills", []))
    }
    for row in monster_data_list
])
print(len(monster_data_list))
df_crawled.head()

In [ ]:
# Nối df_crawled vào đuôi df_linkedin
df_combined = pd.concat([df_linkedin, df_crawled], ignore_index=True)

# Kiểm tra lại số lượng và dữ liệu sau khi nối
print(f"Số dòng df_linkedin: {len(df_linkedin)}")
print(f"Số dòng df_crawled: {len(df_crawled)}")
print(f"Tổng số dòng df_combined: {len(df_combined)}")

df_combined.tail() 

In [ ]:
import io, json
import pandas as pd
from multiprocessing import Pool, cpu_count

bucket = "bigdata-nhom6"

df_meta = pd.read_csv(client.get_object(bucket, "bronze/kaggle/linkedin_meta.csv"))
df_skills = pd.read_csv(client.get_object(bucket, "bronze/kaggle/linkedin_skills.csv"))
df_linkedin = pd.merge(df_meta, df_skills, on="job_link")[['job_link', 'job_title', 'job_skills']]

monster_data_list = []
objects = client.list_objects(bucket, prefix="bronze/crawled/", recursive=True)

for obj in objects:
    if obj.object_name.endswith('.json'):
        response_json = client.get_object(bucket, obj.object_name)
        monster_data_list.extend(json.loads(response_json.read().decode('utf-8'))) 

df_monster = pd.DataFrame(monster_data_list)
if not df_monster.empty:
    df_monster['job_skills'] = df_monster['skills'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
    df_monster = df_monster.rename(columns={'title': 'job_title', 'link': 'job_link'})[['job_link', 'job_title', 'job_skills']]
else:
    df_monster = pd.DataFrame(columns=['job_link', 'job_title', 'job_skills'])

df_silver = pd.concat([df_linkedin, df_monster], ignore_index=True).dropna(subset=['job_title', 'job_skills'])

with Pool(cpu_count()) as p:
    df_silver['job_title'] = p.map(process_text_wrapper, df_silver['job_title'])
    df_silver['job_skills'] = p.map(process_text_wrapper, df_silver['job_skills'])

df_silver['title_skills'] = df_silver['job_title'] + " " + df_silver['job_skills']

csv_bytes = df_silver.to_csv(index=False).encode('utf-8')
client.put_object(
    bucket_name=bucket,
    object_name="silver/Silver_Jobs_Cleaned.csv",
    data=io.BytesIO(csv_bytes),
    length=len(csv_bytes),
    content_type='application/csv'
)

# 3 Gold

## 3.1 Vectorization & Embeddings

In [ ]:
df_gold = pd.read_csv('Silver_Jobs_Cleaned.csv').dropna(subset=['title_skills'])

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=100, max_df=0.8, stop_words='english')
X = vectorizer.fit_transform(df_gold['title_skills'])

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = df_gold["title_skills"].tolist()
pool = model.start_multi_process_pool(target_devices=["cuda:0", "cuda:1"]) 

with open("embeddings.npy", "wb") as f:
    for i in tqdm(range(0, len(texts), 100000)):
        chunk = texts[i:i+100000]
        emb = model.encode_multi_process(chunk, pool, batch_size=2048, normalize_embeddings=True)
        np.save(f, emb)

model.stop_multi_process_pool(pool)

## 3.2 FAISS Index & Recommender

In [ ]:
index = faiss.IndexFlatIP(384) 
with open("embeddings.npy", "rb") as f:
    while True:
        try:
            index.add(np.load(f))
        except (ValueError, EOFError):
            break

def recommend_and_analyze(user_cv_text, user_skills_list):
    cv_clean = process_text_wrapper(user_cv_text)
    query_vector = model.encode([cv_clean], normalize_embeddings=True)
    D, I = index.search(query_vector, k=10)
    
    display_df = df_gold.iloc[I[0]][["job_title", "job_skills"]]
    
    user_skills_set = set([s.lower() for s in user_skills_list])
    missing = []
    for idx in I[0]:
        job_skills = set(df_gold.iloc[idx]["job_skills"].lower().split(", "))
        missing += list(job_skills - user_skills_set)
        
    return display_df, Counter(missing).most_common(10)

# Ví dụ test:
# df_result, missing_skills = recommend_and_analyze("Data Scientist", ["python", "sql"])
# print(missing_skills)